Cell 1: Install + Import

In [ ]:
import pandas as pd
import numpy as np
import re, math, json
from collections import defaultdict, Counter
from nltk.stem import PorterStemmer

Cell 2: Mount Drive + Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset_path = '/content/drive/MyDrive/Data Mining/DM Project/flipkart_com-ecommerce_sample.csv'
df = pd.read_csv(dataset_path)

df.head()

,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23 +0000,http://www.flipkart.com/fabhomedecor-fabric-do...,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",SBEEH3QGU7MFYJFY,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,No rating available,No rating available,FabHomeDecor,"{""product_specification""=>[{""key""=>""Installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23 +0000,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",SHOEH4GRSUBJGZXE,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,Key Features of AW Bellies Sandals Wedges Heel...,No rating available,No rating available,AW,"{""product_specification""=>[{""key""=>""Ideal For""..."
3,0973b37acd0c664e3de26e97e5571454,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2F6HUZMQ6SJ,699.0,267.0,"[""http://img5a.flixcart.com/image/short/6/2/h/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,2016-03-25 22:59:23 +0000,http://www.flipkart.com/sicons-all-purpose-arn...,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",PSOEH3ZYDMSYARJ5,220.0,210.0,"[""http://img5a.flixcart.com/image/pet-shampoo/...",False,Specifications of Sicons All Purpose Arnica Do...,No rating available,No rating available,Sicons,"{""product_specification""=>[{""key""=>""Pet Type"",..."


Cell 3: Select Useful Columns

In [ ]:
df = df[
    [
        'product_name',
        'product_category_tree',
        'description',
        'retail_price',
        'discounted_price',
        'brand',
        'product_url',
        'image'
    ]
].copy()

df.fillna('', inplace=True)
df = df.reset_index(drop=True)
df['doc_id'] = df.index

df.head()

/tmp/ipykernel_20898/1069712503.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace=True)


,product_name,product_category_tree,description,retail_price,discounted_price,brand,product_url,image,doc_id
0,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",Key Features of Alisha Solid Women's Cycling S...,999.0,379.0,Alisha,http://www.flipkart.com/alisha-solid-women-s-c...,"[""http://img5a.flixcart.com/image/short/u/4/a/...",0
1,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",FabHomeDecor Fabric Double Sofa Bed (Finish Co...,32157.0,22646.0,FabHomeDecor,http://www.flipkart.com/fabhomedecor-fabric-do...,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",1
2,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",Key Features of AW Bellies Sandals Wedges Heel...,999.0,499.0,AW,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",2
3,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",Key Features of Alisha Solid Women's Cycling S...,699.0,267.0,Alisha,http://www.flipkart.com/alisha-solid-women-s-c...,"[""http://img5a.flixcart.com/image/short/6/2/h/...",3
4,Sicons All Purpose Arnica Dog Shampoo,"[""Pet Supplies >> Grooming >> Skin & Coat Care...",Specifications of Sicons All Purpose Arnica Do...,220.0,210.0,Sicons,http://www.flipkart.com/sicons-all-purpose-arn...,"[""http://img5a.flixcart.com/image/pet-shampoo/...",4


Cell 4: Preprocessing Function

In [ ]:
stemmer = PorterStemmer()

stopwords = {
    'the','is','a','an','and','or','of','to','in','for','with','on','by','this',
    'that','it','as','at','from','be','are','was','were','will','you','your'
}

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    tokens = [stemmer.stem(word) for word in tokens if word not in stopwords]
    return tokens

Cell 5: Create Search Text

In [ ]:
df['search_text'] = (
    df['product_name'].astype(str) + ' ' +
    df['brand'].astype(str) + ' ' +
    df['product_category_tree'].astype(str) + ' ' +
    df['description'].astype(str)
)

df['tokens'] = df['search_text'].apply(preprocess)

df[['doc_id', 'product_name', 'tokens']].head()

,doc_id,product_name,tokens
0,0,Alisha Solid Women's Cycling Shorts,"[alisha, solid, women, s, cycl, short, alisha,..."
1,1,FabHomeDecor Fabric Double Sofa Bed,"[fabhomedecor, fabric, doubl, sofa, bed, fabho..."
2,2,AW Bellies,"[aw, belli, aw, footwear, women, s, footwear, ..."
3,3,Alisha Solid Women's Cycling Shorts,"[alisha, solid, women, s, cycl, short, alisha,..."
4,4,Sicons All Purpose Arnica Dog Shampoo,"[sicon, all, purpos, arnica, dog, shampoo, sic..."


Cell 6: Build Inverted Index

In [ ]:
inverted_index = defaultdict(dict)

for _, row in df.iterrows():
    doc_id = int(row['doc_id'])
    tokens = row['tokens']

    for position, term in enumerate(tokens):
        if doc_id not in inverted_index[term]:
            inverted_index[term][doc_id] = {
                'frequency': 0,
                'positions': []
            }
        inverted_index[term][doc_id]['frequency'] += 1
        inverted_index[term][doc_id]['positions'].append(position)

print("Total terms in index:", len(inverted_index))

Total terms in index: 22042


Cell 7: Boolean Query Processing

In [ ]:
def get_docs_for_term(term):
    term = stemmer.stem(term.lower())
    return set(inverted_index.get(term, {}).keys())

def boolean_search(query):
    query = query.lower()

    if ' and ' in query:
        terms = query.split(' and ')
        result = get_docs_for_term(terms[0])
        for term in terms[1:]:
            result = result.intersection(get_docs_for_term(term))
        return result

    elif ' or ' in query:
        terms = query.split(' or ')
        result = set()
        for term in terms:
            result = result.union(get_docs_for_term(term))
        return result

    else:
        terms = preprocess(query)
        if not terms:
            return set()

        result = get_docs_for_term(terms[0])
        for term in terms[1:]:
            result = result.intersection(get_docs_for_term(term))
        return result

Cell 8: TF-IDF Ranking

In [ ]:
N = len(df)

def calculate_score(query, doc_id):
    query_terms = preprocess(query)
    score = 0

    for term in query_terms:
        if term in inverted_index and doc_id in inverted_index[term]:
            tf = inverted_index[term][doc_id]['frequency']
            df_term = len(inverted_index[term])
            idf = math.log((N + 1) / (df_term + 1)) + 1
            score += tf * idf

    return score

Cell 9: Snippet + Highlight

In [ ]:
def make_snippet(text, query, length=180):
    text = str(text)
    query_terms = query.lower().replace(' and ', ' ').replace(' or ', ' ').split()

    lower_text = text.lower()
    start = 0

    for term in query_terms:
        pos = lower_text.find(term)
        if pos != -1:
            start = max(0, pos - 50)
            break

    snippet = text[start:start+length]

    for term in query_terms:
        snippet = re.sub(f'({re.escape(term)})', r'**\1**', snippet, flags=re.IGNORECASE)

    return snippet + "..."

Cell 10: Main Search Function

In [ ]:
def search_engine(query, top_k=10):
    matched_docs = boolean_search(query)

    ranked_results = []

    for doc_id in matched_docs:
        score = calculate_score(query, doc_id)
        ranked_results.append((doc_id, score))

    ranked_results = sorted(ranked_results, key=lambda x: x[1], reverse=True)

    results = []

    for doc_id, score in ranked_results[:top_k]:
        row = df.iloc[doc_id]
        results.append({
            'Product Name': row['product_name'],
            'Brand': row['brand'],
            'Category': row['product_category_tree'],
            'Retail Price': row['retail_price'],
            'Discounted Price': row['discounted_price'],
            'Score': round(score, 4),
            'Snippet': make_snippet(row['description'], query)
        })

    return pd.DataFrame(results)

Cell 11: Test Search

In [ ]:
search_engine("samsung phone", top_k=10)

,Product Name,Brand,Category,Retail Price,Discounted Price,Score,Snippet
0,AdroitZ Premium Phone Socket Holder For Samsun...,AdroitZ,"[""Mobiles & Accessories >> Mobile Accessories ...",599.0,164.0,75.0975,atures of AdroitZ Premium **Phone** Socket Hol...
1,AdroitZ Premium Phone Socket Holder For Samsun...,AdroitZ,"[""Mobiles & Accessories >> Mobile Accessories ...",599.0,164.0,75.0975,atures of AdroitZ Premium **Phone** Socket Hol...
2,AdroitZ Premium Phone Socket Holder For Samsun...,AdroitZ,"[""Mobiles & Accessories >> Mobile Accessories ...",599.0,164.0,75.0975,atures of AdroitZ Premium **Phone** Socket Hol...
3,Karp Bracelet Necklace Pearl wrist line data p...,Karp,"[""Computers >> Laptop Accessories >> USB Gadge...",1099.0,599.0,65.5873,le beaded fashion bracelet USB charging cable ...
4,Generix OTG for Samsung Galaxy Tab 3 OTG Cable,Generix,"[""Mobiles & Accessories >> Mobile Accessories ...",499.0,199.0,62.2533,Key Features of Generix OTG for **Samsung** Ga...
5,kits kart Pouch for Samsung Galaxy Grand Prime...,kits kart,"[""Mobiles & Accessories >> Mobile Accessories ...",908.0,454.0,52.7432,Key Features of kits kart Pouch for **Samsung*...
6,kits kart Pouch for Samsung Galaxy Grand Neo i...,kits kart,"[""Mobiles & Accessories >> Mobile Accessories ...",908.0,454.0,52.7432,Key Features of kits kart Pouch for **Samsung*...
7,kits kart Pouch for Samsung Z3,kits kart,"[""Mobiles & Accessories >> Mobile Accessories ...",908.0,454.0,52.7432,Key Features of kits kart Pouch for **Samsung*...
8,99Gems LR GOLD 5 in 1 MOBILE / MP3 USB USB Cable,99Gems,"[""Computers >> Laptop Accessories >> USB Gadge...",199.0,149.0,50.3929,Smart **Phone**s And Some Other Mobile **Phone...
9,Shop Swipe Anti-Radiation Retro Style Wired He...,Shop Swipe,"[""Mobiles & Accessories >> Mobile Accessories ...",1200.0,559.0,50.3929,Style 3.5mm Jack Wired Handset **Phone** I**p...


Cell 12: Test AND Query

In [ ]:
search_engine("samsung and phone", top_k=10)

,Product Name,Brand,Category,Retail Price,Discounted Price,Score,Snippet
0,AdroitZ Premium Phone Socket Holder For Samsun...,AdroitZ,"[""Mobiles & Accessories >> Mobile Accessories ...",599.0,164.0,75.0975,atures of AdroitZ Premium **Phone** Socket Hol...
1,AdroitZ Premium Phone Socket Holder For Samsun...,AdroitZ,"[""Mobiles & Accessories >> Mobile Accessories ...",599.0,164.0,75.0975,atures of AdroitZ Premium **Phone** Socket Hol...
2,AdroitZ Premium Phone Socket Holder For Samsun...,AdroitZ,"[""Mobiles & Accessories >> Mobile Accessories ...",599.0,164.0,75.0975,atures of AdroitZ Premium **Phone** Socket Hol...
3,Karp Bracelet Necklace Pearl wrist line data p...,Karp,"[""Computers >> Laptop Accessories >> USB Gadge...",1099.0,599.0,65.5873,le beaded fashion bracelet USB charging cable ...
4,Generix OTG for Samsung Galaxy Tab 3 OTG Cable,Generix,"[""Mobiles & Accessories >> Mobile Accessories ...",499.0,199.0,62.2533,Key Features of Generix OTG for **Samsung** Ga...
5,kits kart Pouch for Samsung Galaxy Grand Prime...,kits kart,"[""Mobiles & Accessories >> Mobile Accessories ...",908.0,454.0,52.7432,Key Features of kits kart Pouch for **Samsung*...
6,kits kart Pouch for Samsung Galaxy Grand Neo i...,kits kart,"[""Mobiles & Accessories >> Mobile Accessories ...",908.0,454.0,52.7432,Key Features of kits kart Pouch for **Samsung*...
7,kits kart Pouch for Samsung Z3,kits kart,"[""Mobiles & Accessories >> Mobile Accessories ...",908.0,454.0,52.7432,Key Features of kits kart Pouch for **Samsung*...
8,99Gems LR GOLD 5 in 1 MOBILE / MP3 USB USB Cable,99Gems,"[""Computers >> Laptop Accessories >> USB Gadge...",199.0,149.0,50.3929,Smart **Phone**s And Some Other Mobile **Phone...
9,Shop Swipe Anti-Radiation Retro Style Wired He...,Shop Swipe,"[""Mobiles & Accessories >> Mobile Accessories ...",1200.0,559.0,50.3929,Style 3.5mm Jack Wired Handset **Phone** I**p...


Cell 13: Test OR Query

In [ ]:
search_engine("shoes or watch", top_k=10)

,Product Name,Brand,Category,Retail Price,Discounted Price,Score,Snippet
0,Fastrack 9912PP15 Tees Analog Watch - For Men...,,"[""Watches >> Wrist Watches >> Fastrack Wrist W...",750.0,750.0,131.6748,Fastrack 9912PP15 Tees Analog **Watch** - For...
1,Maxima 19884LMLI Swarovski Analog Watch - For...,,"[""Watches >> Wrist Watches >> Maxima Wrist Wat...",,,131.6748,Maxima 19884LMLI Swarovski Analog **Watch** -...
2,N Five Running Shoes,N Five,"[""Footwear >> Kids' & Infant Footwear >> For B...",2398.0,959.0,127.5590,Key Features of N Five Running **Shoes** Foot ...
3,Anno Dominii ADWB0000139 Watch Box,Anno Dominii,"[""Watches >> Watch Accessories >> Watch Boxes ...",1499.0,1499.0,118.0532,Key Features of Anno Dominii ADWB0000139 **Wat...
4,V-Luma W600 Fashion Analog Watch - For Men,V-Luma,"[""Watches >> Wrist Watches >> V-Luma Wrist Wat...",1599.0,699.0,108.9722,Key Features of V-Luma W600 Fashion Analog **W...
5,"Romex Ultimate Urban Analog Watch - For Boys,...",,"[""Watches >> Wrist Watches >> Romex Wrist Watc...",1199.0,499.0,86.2697,Romex Ultimate Urban Analog **Watch** - For B...
6,Lee Parke Running Shoes,Lee Parke,"[""Footwear >> Men's Footwear >> Sports Shoes >...",999.0,499.0,68.0315,Key Features of Lee Parke Running **Shoes** Ca...
7,Lee Parke Walking Shoes,Lee Parke,"[""Footwear >> Men's Footwear >> Sports Shoes >...",999.0,999.0,68.0315,Key Features of Lee Parke Walking **Shoes** Ca...
8,RED HORSE Canvas Shoes,RED HORSE,"[""Footwear >> Kids' & Infant Footwear >> For G...",1999.0,899.0,63.7795,Key Features of RED HORSE Canvas **Shoes** Mat...
9,Desire PT-378 Analog Watch - For Women,Desire,"[""Watches >> Wrist Watches >> Desire Wrist Wat...",999.0,699.0,63.5671,Key Features of Desire PT-378 Analog **Watch**...


Cell 14: Save Index for Submission

In [ ]:
index_to_save = {
    term: {
        str(doc_id): data
        for doc_id, data in docs.items()
    }
    for term, docs in inverted_index.items()
}

with open('inverted_index.json', 'w') as f:
    json.dump(index_to_save, f)

df.drop(columns=['tokens']).to_csv('processed_products.csv', index=False)

print("Files saved successfully.")

Files saved successfully.


In [ ]:
!ls

app.py	inverted_index.json	requirements.txt
drive	processed_products.csv	sample_data


Cell 15

In [ ]:
%%writefile app.py
import gradio as gr
import pandas as pd
import json, re, math, time, ast
from collections import defaultdict
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

STOPWORDS = {
    'the','is','a','an','and','or','of','to','in','for','with','on','by','this',
    'that','it','as','at','from','be','are','was','were','will','you','your'
}

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    return [stemmer.stem(word) for word in tokens if word not in STOPWORDS]

df = pd.read_csv("processed_products.csv")

with open("inverted_index.json", "r") as f:
    raw_index = json.load(f)

inverted_index = defaultdict(dict)
for term, docs in raw_index.items():
    for doc_id, data in docs.items():
        inverted_index[term][int(doc_id)] = data

N = len(df)

def get_docs_for_term(term):
    term = stemmer.stem(str(term).lower())
    return set(inverted_index.get(term, {}).keys())

def boolean_search(query):
    query = query.lower().strip()

    if " and " in query:
        terms = query.split(" and ")
        result = get_docs_for_term(terms[0])
        for term in terms[1:]:
            result = result.intersection(get_docs_for_term(term))
        return result

    elif " or " in query:
        terms = query.split(" or ")
        result = set()
        for term in terms:
            result = result.union(get_docs_for_term(term))
        return result

    else:
        terms = preprocess(query)
        if not terms:
            return set()

        result = get_docs_for_term(terms[0])
        for term in terms[1:]:
            result = result.intersection(get_docs_for_term(term))
        return result

def calculate_score(query, doc_id):
    query_terms = preprocess(query)
    score = 0

    for term in query_terms:
        if term in inverted_index and doc_id in inverted_index[term]:
            tf = inverted_index[term][doc_id]["frequency"]
            df_term = len(inverted_index[term])
            idf = math.log((N + 1) / (df_term + 1)) + 1
            score += tf * idf

    return score

def highlight(text, query):
    text = str(text)
    query_terms = query.lower().replace(" and ", " ").replace(" or ", " ").split()

    for term in query_terms:
        text = re.sub(
            f"({re.escape(term)})",
            r"<mark>\1</mark>",
            text,
            flags=re.IGNORECASE
        )
    return text

def get_first_image(image_value):
    try:
        images = ast.literal_eval(str(image_value))
        if isinstance(images, list) and len(images) > 0:
            return images[0]
    except:
        pass
    return ""

def search_products(query, top_k):
    start_time = time.time()

    if query.strip() == "":
        return "<h3>Please enter a search query.</h3>"

    matched_docs = boolean_search(query)

    ranked = []
    for doc_id in matched_docs:
        score = calculate_score(query, doc_id)
        ranked.append((doc_id, score))

    ranked = sorted(ranked, key=lambda x: x[1], reverse=True)
    search_time = round(time.time() - start_time, 4)

    if not ranked:
        return f"<h3>No result found for: {query}</h3><p>Try another keyword.</p>"

    html = f"""
    <div style="font-family: Arial;">
        <p style="color:gray;">Found {len(ranked)} results in {search_time} seconds</p>
    """

    for doc_id, score in ranked[:top_k]:
        row = df.iloc[doc_id]

        title = highlight(row.get("product_name", ""), query)
        desc = highlight(str(row.get("description", ""))[:280], query)
        image_url = get_first_image(row.get("image", ""))
        product_url = row.get("product_url", "#")

        image_html = ""
        if image_url:
            image_html = f"""
            <img src="{image_url}"
                 style="width:120px; height:120px; object-fit:contain; border:1px solid #eee; border-radius:8px; margin-right:18px;">
            """

        html += f"""
        <div style="
            display:flex;
            gap:18px;
            border:1px solid #ddd;
            padding:18px;
            margin:15px 0;
            border-radius:12px;
            background:white;
        ">
            <div>{image_html}</div>

            <div style="flex:1;">
                <h3 style="color:#1a0dab; margin-bottom:8px;">{title}</h3>
                <p><b>Brand:</b> {row.get("brand", "")}</p>
                <p>
                    <b>Retail Price:</b> ₹{row.get("retail_price", "")}
                    |
                    <b>Discounted Price:</b> ₹{row.get("discounted_price", "")}
                </p>
                <p><b>TF-IDF Score:</b> {round(score, 4)}</p>
                <p>{desc}...</p>
                <a href="{product_url}" target="_blank"
                   style="display:inline-block; padding:8px 14px; background:#1a73e8; color:white; text-decoration:none; border-radius:6px;">
                   View Product
                </a>
            </div>
        </div>
        """

    html += "</div>"
    return html

css = """
.gradio-container {
    max-width: 980px !important;
    margin: auto !important;
}
h1 {
    text-align: center;
}
"""

with gr.Blocks(css=css, title="E-Commerce Search Engine") as demo:
    gr.HTML("""
    <h1>🛒 ShopSearch</h1>
    <p style="text-align:center; color:gray;">
    E-Commerce Product Search Engine using Custom Inverted Index and TF-IDF Ranking
    </p>
    """)

    query = gr.Textbox(
        label="Search Product",
        placeholder="Search for samsung phone, shoes or watch..."
    )

    top_k = gr.Dropdown(
        choices=[5, 10, 20, 50],
        value=10,
        label="Number of Results"
    )

    search_btn = gr.Button("Search")
    output = gr.HTML()

    search_btn.click(
        fn=search_products,
        inputs=[query, top_k],
        outputs=output
    )

demo.launch()

Overwriting app.py


In [ ]:
!grep -n "View Product" app.py
!grep -n "get_first_image" app.py

161:                   View Product
89:def get_first_image(image_value):
127:        image_url = get_first_image(row.get("image", ""))


In [ ]:
from google.colab import files
files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls

app.py	inverted_index.json	requirements.txt
drive	processed_products.csv	sample_data


Cell 16

In [ ]:
from google.colab import files

files.download("app.py")
files.download("processed_products.csv")
files.download("inverted_index.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cell 17

In [ ]:
%%writefile requirements.txt
gradio
pandas
nltk
numpy

Writing requirements.txt


Cell 18

In [ ]:
from google.colab import files

files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls

app.py	inverted_index.json	requirements.txt
drive	processed_products.csv	sample_data
